# LangChain with Azure OpenAI, AI Search, and AI Foundry

This notebook demonstrates integration with Azure AI services using LangChain.

## ⚠️ Security Notice

All sensitive credentials have been moved to `config.txt` for security. Before running this notebook:

1. Copy `config.txt.template` to `config.txt`
2. Fill in your Azure credentials in `config.txt`
3. Verify `config.txt` is in `.gitignore` (it should be)

## Install Required Packages

Run this cell first to install all dependencies:

In [ ]:
%pip install langchain-openai

## Load Configuration

Load Azure credentials and configuration from external file:

In [ ]:
# Load configuration from config.txt file
import os

def load_config(config_file="config.txt"):
    """Load configuration from a key=value text file"""
    config = {}
    with open(config_file, 'r') as f:
        for line in f:
            line = line.strip()
            # Skip comments and empty lines
            if line and not line.startswith('#'):
                if '=' in line:
                    key, value = line.split('=', 1)
                    config[key.strip()] = value.strip()
    return config

# Load all configuration
config = load_config()

print("Configuration loaded successfully")
print(f"Loaded {len(config)} configuration values")

# LangChain with Azure OpenAI

This notebook demonstrates how to use LangChain with Azure OpenAI for chat completions.

## 1. Import Required Libraries

In [ ]:
import os
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

## 2. Configure Azure OpenAI Endpoint and Deployment

In [ ]:
# Load Azure OpenAI configuration from config file
endpoint = config.get('AZURE_OPENAI_ENDPOINT')
deployment = config.get('AZURE_OPENAI_DEPLOYMENT')
api_key = config.get('AZURE_OPENAI_API_KEY')

## 3. Initialize LangChain Azure OpenAI with API Key

In [ ]:
# Initialize LangChain AzureChatOpenAI with API key
llm = AzureChatOpenAI(
    azure_endpoint=endpoint,
    api_key=api_key,
    azure_deployment=deployment,
    api_version=config.get('AZURE_OPENAI_API_VERSION'),
    temperature=0.7,
    max_tokens=13107,
    top_p=0.95,
)

print("Azure OpenAI client initialized")
print(f"Endpoint: {endpoint}")
print(f"Deployment: {deployment}")

## 4. Prepare Chat Messages

In [ ]:
# Prepare messages using LangChain message classes
messages = [
    SystemMessage(content="You are an AI assistant that helps people find information."),
    HumanMessage(content="hello"),
    AIMessage(content="Hello! How can I assist you today?"),
    HumanMessage(content="What can you help me with?")
]

## 5. Create Chat Completion and Display Results

In [ ]:
# Invoke the LangChain model
response = llm.invoke(messages)

# Display the response
print("Response content:")
print(response.content)
print("\n" + "="*50 + "\n")
print("Full response object:")
print(response)

## 6. (Optional) Streaming Response

LangChain supports streaming responses for real-time output:

In [ ]:
# Stream response tokens
print("Streaming response:")
for chunk in llm.stream(messages):
    print(chunk.content, end="", flush=True)
print("\n")

## 7. (Optional) Batch Processing

Process multiple message sets in parallel:

In [ ]:
# Create multiple message sets
batch_messages = [
    [HumanMessage(content="What is 2+2?")],
    [HumanMessage(content="What is the capital of France?")],
    [HumanMessage(content="Tell me a short joke.")]
]

# Process in batch
batch_responses = llm.batch(batch_messages)

# Display results
for i, response in enumerate(batch_responses):
    print(f"Query {i+1}: {batch_messages[i][0].content}")
    print(f"Response: {response.content}\n")

In [ ]:
!pip install openai

In [ ]:
from openai import AzureOpenAI

# Load embeddings configuration from config file
embeddings_endpoint = config.get('AZURE_EMBEDDINGS_ENDPOINT')
model_name = config.get('AZURE_EMBEDDINGS_DEPLOYMENT')
deployment = config.get('AZURE_EMBEDDINGS_DEPLOYMENT')
api_version = config.get('AZURE_EMBEDDINGS_API_VERSION')
api_key = config.get('AZURE_OPENAI_API_KEY')

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=embeddings_endpoint,
    api_key=api_key
)

response = client.embeddings.create(
    input=["first phrase","second phrase","third phrase"],
    model=deployment
)

for item in response.data:
    length = len(item.embedding)
    print(
        f"data[{item.index}]: length={length}, "
        f"[{item.embedding[0]}, {item.embedding[1]}, "
        f"..., {item.embedding[length-2]}, {item.embedding[length-1]}]"
    )
print(response.usage)

## 8. Azure AI Search with Vector Store

Now let's fetch Wikipedia articles about famous soccer players, create embeddings, and upload them to Azure AI Search.

In [ ]:
%pip install azure-search-documents wikipedia langchain-community

### Fetch Wikipedia Articles for Famous Soccer Players

In [ ]:
import wikipedia

# List of 20 famous soccer players
soccer_players = [
    "Lionel Messi",
    "Cristiano Ronaldo",
    "Pelé",
    "Diego Maradona",
    "Zinedine Zidane",
    "Johan Cruyff",
    "Franz Beckenbauer",
    "Ronaldo (Brazilian footballer)",
    "Ronaldinho",
    "Thierry Henry",
    "Andrés Iniesta",
    "Xavi",
    "Neymar",
    "Kylian Mbappé",
    "Robert Lewandowski",
    "Mohamed Salah",
    "Kevin De Bruyne",
    "Luka Modrić",
    "Erling Haaland",
    "Sergio Ramos"
]

# Fetch Wikipedia content
documents = []
for player in soccer_players:
    try:
        print(f"Fetching article for {player}...")
        page = wikipedia.page(player, auto_suggest=False)
        documents.append({
            "id": player.replace(" ", "_").replace("(", "").replace(")", ""),
            "title": player,
            "content": page.content,
            "url": page.url
        })
        print(f"Fetched {len(page.content)} characters for {player}")
    except Exception as e:
        print(f"Error fetching {player}: {e}")

print(f"\nSuccessfully fetched {len(documents)} articles")

### Chunk the Documents

In [ ]:
def chunk_text(text, chunk_size=500, overlap=50):
    """Split text into overlapping chunks"""
    chunks = []
    start = 0
    text_length = len(text)
    
    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap
    
    return chunks

# Chunk all documents
chunked_documents = []
chunk_id = 0

for doc in documents:
    chunks = chunk_text(doc["content"])
    for i, chunk in enumerate(chunks):
        chunked_documents.append({
            "id": f"{doc['id']}_chunk_{i}",
            "chunk_id": str(chunk_id),
            "player_name": doc["title"],
            "content": chunk,
            "url": doc["url"],
            "chunk_index": i
        })
        chunk_id += 1

print(f"Created {len(chunked_documents)} chunks from {len(documents)} documents")
print(f"  Average chunks per document: {len(chunked_documents) / len(documents):.1f}")

### Configure Azure AI Search

In [ ]:
from langchain_community.vectorstores.azuresearch import AzureSearch
from langchain_openai import AzureOpenAIEmbeddings

# Azure AI Search configuration from config file
search_service_name = config.get('AZURE_SEARCH_SERVICE_NAME')
search_admin_key = config.get('AZURE_SEARCH_ADMIN_KEY')
search_endpoint = f"https://{search_service_name}.search.windows.net"
index_name = config.get('AZURE_SEARCH_INDEX_NAME')

# Initialize Azure OpenAI embeddings for LangChain
embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=config.get('AZURE_EMBEDDINGS_ENDPOINT'),
    api_key=config.get('AZURE_OPENAI_API_KEY'),
    azure_deployment=config.get('AZURE_EMBEDDINGS_DEPLOYMENT'),
    api_version=config.get('AZURE_EMBEDDINGS_API_VERSION')
)

# Create Azure Search vector store
vector_store = AzureSearch(
    azure_search_endpoint=search_endpoint,
    azure_search_key=search_admin_key,
    index_name=index_name,
    embedding_function=embeddings.embed_query,
    additional_search_client_options={"api_version": "2024-05-01-preview"}
)

print("Azure Search Vector Store initialized")
print(f"Endpoint: {search_endpoint}")
print(f"Index name: {index_name}")

### Upload Documents to Vector Store

In [ ]:
from langchain_core.documents import Document

# Convert chunked documents to LangChain Document objects
langchain_docs = []
for doc in chunked_documents:
    langchain_docs.append(
        Document(
            page_content=doc["content"],
            metadata={
                "player_name": doc["player_name"],
                "url": doc["url"],
                "chunk_index": doc["chunk_index"],
                "chunk_id": doc["chunk_id"]
            }
        )
    )

print(f"Uploading {len(langchain_docs)} documents to vector store...")

# Upload documents in batches
batch_size = 100
total_uploaded = 0

for i in range(0, len(langchain_docs), batch_size):
    batch = langchain_docs[i:i+batch_size]
    try:
        vector_store.add_documents(documents=batch)
        total_uploaded += len(batch)
        print(f"Batch {i//batch_size + 1}: Uploaded {len(batch)} documents")
    except Exception as e:
        print(f"Error uploading batch {i//batch_size + 1}: {e}")

print(f"\nSuccessfully uploaded {total_uploaded} documents to vector store")
print(f"  Index: {index_name}")
print(f"  Endpoint: {search_endpoint}")

### Helper Function: Fetch Context from Vector Store

In [ ]:
def fetch_vector_context(
    query: str,
    k: int = 5,
    player_filter: str = None,
    max_content_length: int = 500,
    return_raw_results: bool = False
):
    """
    Fetch relevant context from the vector store based on a query.
    
    Args:
        query: The search query string
        k: Number of results to retrieve (default: 5)
        player_filter: Optional player name to filter results (e.g., "Lionel Messi")
        max_content_length: Maximum length of content per result in context (default: 500)
        return_raw_results: If True, return raw results with scores (default: False)
    
    Returns:
        If return_raw_results is False: formatted context string
        If return_raw_results is True: tuple of (context_string, results_list)
    """
    # Perform vector search
    results = vector_store.similarity_search_with_score(query, k=k)
    
    # Apply player filter if specified
    if player_filter:
        results = [
            (doc, score) for doc, score in results 
            if doc.metadata.get('player_name') == player_filter
        ]
    
    # Format results into context
    context_parts = []
    for doc, score in results:
        player_name = doc.metadata.get('player_name', 'Unknown')
        content = doc.page_content[:max_content_length]
        if len(doc.page_content) > max_content_length:
            content += "..."
        context_parts.append(f"{player_name}: {content}")
    
    context = "\n\n".join(context_parts)
    
    if return_raw_results:
        return context, results
    return context

print("Vector context fetching function defined")
print("Usage: fetch_vector_context(query, k=5, player_filter=None)")

### Example: Basic Vector Search with Function

In [ ]:
# Example: Using the vector context function with LLM
query = "Who is the underrated finisher"

# Fetch context from vector store
context = fetch_vector_context(query, k=5, max_content_length=500)

# Create prompt with context
prompt = f"""You are a helpful assistant that provides information about soccer players based on the context obtained from a vector search. Use the following information to answer the question:

User Query: {query}

Context:
{context}
"""

# Get LLM response
response = llm.invoke([SystemMessage(content=prompt)])

print("Query:", query)
print("\nAnswer:")
print(response.content)

## 9. Azure AI Foundry Agent

Connect to an Azure AI Foundry agent using device code authentication. Run the authentication cell once, then reuse the cached credential in subsequent cells.

In [ ]:
# One-time authentication: Device Code Login
# Run this cell once to authenticate. The credential will be cached for future cells.

from azure.identity import DeviceCodeCredential

# Configure tenant and scope for Azure AI from config file
tenant_id = config.get('AZURE_TENANT_ID')
ai_scope = "https://ai.azure.com/.default"

# Create credential with device code flow
# This will prompt you once with a device code, then cache the token
azure_credential = DeviceCodeCredential(
    tenant_id=tenant_id,
    client_id="04b07795-8ddb-461a-bbee-02f9e1bf7b46"  # Azure CLI public client ID
)

# Pre-authenticate to cache the token (this triggers the device code prompt)
print("Authenticating with device code...")
print("=" * 80)
token = azure_credential.get_token(ai_scope)
print("\nAuthentication successful")
print("Credential cached in variable 'azure_credential'")
print(f"Token valid until: {token.expires_on}")
print("\nYou can now use 'azure_credential' in subsequent cells without re-authenticating.")

In [ ]:
# Use the cached credential from the previous cell
# No authentication prompt - uses the token cached above

from azure.ai.projects import AIProjectClient

# Load Azure AI Foundry configuration from config file
endpoint = config.get('AZURE_AI_FOUNDRY_ENDPOINT')

# Use the credential we authenticated in the previous cell
project_client = AIProjectClient(
    endpoint=endpoint,
    credential=azure_credential,  # Reusing cached credential
)

my_agent = config.get('AZURE_AI_AGENT_NAME')
my_version = config.get('AZURE_AI_AGENT_VERSION')

openai_client = project_client.get_openai_client()

# Reference the agent to get a response
response = openai_client.responses.create(
    input=[{"role": "user", "content": "Who is the current Premier League top scorer?"}],
    extra_body={"agent_reference": {"name": my_agent, "version": my_version, "type": "agent_reference"}},
)

print(f"Response output: {response.output_text}")

### Example: Reusing Cached Credential

You can reuse the `azure_credential` from the authentication cell for multiple queries without re-authenticating:

In [ ]:
# Example: Another query using the same cached credential
# No authentication needed - the credential is already cached

response2 = openai_client.responses.create(
    input=[{"role": "user", "content": "What are the key strategies for a successful counterattack in soccer?"}],
    extra_body={"agent_reference": {"name": my_agent, "version": my_version, "type": "agent_reference"}},
)

print("Query: What are the key strategies for a successful counterattack in soccer?")
print(f"\nResponse: {response2.output_text}")